# Blank-Image Control: Zero-shot VQA on RadImageNet-VQA: MedGemma 4B-it
**Blank-Image Control Experiment:** Evaluates the model with completely uniform gray images (RGB 128, 128, 128) dynamically matching each original image's dimensions.
Closed-ended (yes/no) and multiple-choice questions only. 500 questions, 5 groups x 100, seed 42.
This ablation control tests whether diagnostic performance and answer biases are grounded in visual signals or driven by textual/linguistic priors.

## 1. Setup and Hugging Face login

In [ ]:
!pip install -q -U transformers accelerate qwen-vl-utils datasets scikit-learn

from huggingface_hub import login
try:                                    # Colab
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except ImportError:                     # Kaggle
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
import os, re, math, json
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image

MODEL_ID = "google/medgemma-4b-it"
TAG = "medgemma_4b"                        # used in output file names
SEED = 42
MAX_NEW_TOKENS = 16                       # answer is one letter or yes/no
OUT_CSV = f"vqa_blank_results_{TAG}.csv"
SCORES_CSV = f"vqa_blank_scores_{TAG}.csv"
SAMPLE_CSV = "vqa_sample.csv"             # same file for all four models
torch.manual_seed(SEED)

## 2. Load the data (closed + multiple_choice only)

In [ ]:
ds = load_dataset("raidium/RadImageNet-VQA", "benchmark", split="test")

meta = ds.remove_columns("image").to_pandas()
meta["content_type"] = meta["metadata"].apply(lambda m: m["content_type"])
meta["anatomy"]      = meta["metadata"].apply(lambda m: m["location"])
meta["question_id"]  = meta["metadata"].apply(lambda m: m["question_id"])
meta["idx"] = np.arange(len(meta))        # row position in ds, used to fetch the image later

GROUPS = {   # (question_type, content_type) -> group name
    ("closed", "anatomy"):             "anatomy_yesno",
    ("closed", "pathology"):           "abnormality_yesno",   # "Does the X look abnormal?"
    ("closed", "pathology_specific"):  "pathology_yesno",
    ("multiple_choice", "anatomy"):    "anatomy_mcq",
    ("multiple_choice", "pathology"):  "pathology_mcq",
}
meta["group"] = [GROUPS.get((q, c)) for q, c in zip(meta.question_type, meta.content_type)]
pool = meta[meta.group.notna()].copy()    # drops the 2,000 open-ended questions
pool["answer"] = pool["answer"].str.strip()
pool.loc[pool.question_type == "closed", "answer"] = pool["answer"].str.lower()

print(len(pool), "questions in the pool")
print(pool.groupby("group").size())

## 3. Load and validate evaluation sample (vqa_sample.csv)
Loads the mandatory 500 frozen sample items (100 per group; each yes/no group is 50 `yes` + 50 `no`).
Strict validation verifies that all 500 indices exist and their `question_id` and `group` match the dataset.

In [ ]:
# Make vqa_sample.csv mandatory (do not silently regenerate)
if not os.path.exists(SAMPLE_CSV):
    raise FileNotFoundError(
        f"Mandatory sample file '{SAMPLE_CSV}' not found! "
        "Please provide vqa_sample.csv in the working directory to ensure the exact frozen 500 samples are evaluated."
    )

saved_sample = pd.read_csv(SAMPLE_CSV)

# Validate that all 500 indices exist
if len(saved_sample) != 500:
    raise ValueError(f"Expected exactly 500 samples in {SAMPLE_CSV}, but found {len(saved_sample)}.")

pool_indices = set(pool["idx"])
missing_indices = set(saved_sample["idx"]) - pool_indices
if missing_indices:
    raise ValueError(f"Validation failed: {len(missing_indices)} indices from {SAMPLE_CSV} are missing in the dataset pool!")

# Merge to compare question_id and group values
merged = saved_sample.merge(
    pool[["idx", "question_id", "group"]],
    on="idx",
    how="inner",
    suffixes=("_expected", "_actual")
)

if len(merged) != 500:
    raise ValueError(f"Validation failed: could only match {len(merged)}/500 items from {SAMPLE_CSV} in the pool.")

mismatched_qid = merged[merged["question_id_expected"] != merged["question_id_actual"]]
mismatched_group = merged[merged["group_expected"] != merged["group_actual"]]

if len(mismatched_qid) > 0 or len(mismatched_group) > 0:
    raise ValueError(
        f"Validation failed for {SAMPLE_CSV}: "
        f"{len(mismatched_qid)} question_id mismatches, {len(mismatched_group)} group mismatches!"
    )

# Retain the exact full rows from pool for evaluation
sample = pool[pool.idx.isin(saved_sample["idx"])].sort_values("idx").reset_index(drop=True)
print(f"Validation successful: all {len(sample)} sample indices, question_ids, and groups verified against {SAMPLE_CSV}.")

# smoke test: 1 question per group, never in the evaluation sample
smoke = pool.drop(index=pool.index[pool.idx.isin(sample.idx)]).groupby("group").sample(1, random_state=SEED)

print(len(sample), "evaluation questions;", len(smoke), "smoke-test questions")
print(pd.crosstab(sample.group, sample.answer.where(sample.question_type == "closed", "MCQ")))


## 4. Load the model

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print(model.device)

## 5. Prompts, inference and parsing

In [ ]:
LETTERS = "ABCD"

def build_prompt(row):
    if row["question_type"] == "multiple_choice":
        opts = "\n".join(f"{L}. {c}" for L, c in zip(LETTERS, row["choices"]))
        return ("You are a medical assistant. Look at this image and answer the following multiple-choice "
                "question with ONLY the letter of the correct option (e.g. 'B'), nothing else.\n\n"
                f"Question: {row['question']}\nOptions:\n{opts}")
    return ("You are a medical assistant. Look at this image and answer the following question "
            "with ONLY the single word 'Yes' or 'No'.\n\n"
            f"Question: {row['question']}")

def predict(image, prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                             {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to(model.device, dtype=torch.bfloat16)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

def parse_mcq(output):
    m = re.search(r"\b([A-D])\b", output)          # first standalone option letter
    return m.group(1) if m else "unparsed"

def parse_yesno(output):
    m = re.search(r"\b(yes|no)\b", output.lower())  # whichever appears first
    return m.group(1) if m else "unparsed"

def parse(row, output):
    m = re.search(r"<answer>(.*?)</answer>", output, re.S | re.I)   # MedVLM-R1 may answer as <answer>X</answer>
    text = m.group(1) if m else output
    return parse_mcq(text) if row["question_type"] == "multiple_choice" else parse_yesno(text)

def create_blank_gray_image(orig_image, color=(128, 128, 128)):
    """Generate a uniform gray image preserving the original image dimensions (width, height)."""
    return Image.new("RGB", orig_image.size, color=color)

def run_one(row):
    orig_image = ds[int(row["idx"])]["image"].convert("RGB")
    blank_image = create_blank_gray_image(orig_image, color=(128, 128, 128))
    raw = predict(blank_image, build_prompt(row))
    return raw, parse(row, raw)


## 6. Smoke test (5 questions, not in the results)
Check that outputs parse, nothing runs out of memory, and the time per question.

In [ ]:
import time
for _, r in smoke.iterrows():
    t = time.time()
    orig = ds[int(r["idx"])]["image"].convert("RGB")
    raw, pred = run_one(r)
    print(f"[{r.group}] orig_size={orig.size} true={r.answer!r} pred={pred!r} raw={raw!r} ({time.time()-t:.1f}s)")


## 7. Run the 500 questions (resumes if the session restarts)

In [ ]:
done = pd.read_csv(OUT_CSV) if os.path.exists(OUT_CSV) else pd.DataFrame()
rows = done.to_dict("records")
seen = set(done["idx"]) if len(done) else set()

for _, r in tqdm(sample.iterrows(), total=len(sample)):
    if r["idx"] in seen:
        continue
    raw, pred = run_one(r)
    rows.append({"idx": r["idx"], "question_id": r["question_id"], "group": r["group"],
                 "anatomy": r["anatomy"], "question": r["question"],
                 "true_answer": r["answer"], "predicted_answer": pred, "raw_output": raw})
    if len(rows) % 50 == 0:
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

results = pd.DataFrame(rows)
results.to_csv(OUT_CSV, index=False)
print(len(results), "rows saved to", OUT_CSV)

## 8. Score
Unparsed outputs count as wrong and stay in the denominator.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (0, 0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h

results["correct"] = results.true_answer == results.predicted_answer
n, k = len(results), int(results.correct.sum())
lo, hi = wilson(k, n)
print(f"Overall accuracy: {k}/{n} = {k/n:.3f}  (95% CI {lo:.3f}-{hi:.3f})")
print(f"Parsed cleanly:   {(results.predicted_answer != 'unparsed').sum()}/{n}\n")

by_group = results.groupby("group").correct.agg(["sum", "count", "mean"]).rename(columns={"sum": "correct", "count": "n", "mean": "accuracy"})
print(by_group.round(3), "\n")
print(results.groupby("anatomy").correct.agg(["count", "mean"]).round(3).rename(columns={"count": "n", "mean": "accuracy"}), "\n")

yn = results[results.group.str.endswith("yesno")]
print(classification_report(yn.true_answer, yn.predicted_answer, labels=["yes", "no"], zero_division=0))
print(pd.DataFrame(confusion_matrix(yn.true_answer, yn.predicted_answer, labels=["yes", "no", "unparsed"]),
                   index=["true yes", "true no", "true unparsed"], columns=["pred yes", "pred no", "pred unparsed"]))

by_group.to_csv(SCORES_CSV)
print(f"Scores saved to {SCORES_CSV}")


## 9. Misclassified cases

In [ ]:
wrong = results[~results.correct][["group", "question", "true_answer", "predicted_answer", "raw_output"]]
print(len(wrong), "wrong")
wrong.head(30)